# 14 - Foundry IQ with Web IQ only

Create a separate knowledge base with exactly one source: basic Web IQ web search. Calls go directly to Azure AI Search, without APIM, Blob, SharePoint, indexes or agents. Notebooks 09 and 10 are not prerequisites.

The source stores the Web IQ API key, so both notebook and Foundry IQ portal queries can authenticate. Retrieval output includes MCP activity, reference URLs and retrieved passages, not just the generated answer.

## Required permissions

| Function | Required access |
|---|---|
| Create/update/delete the knowledge source and knowledge base | Search Service Contributor on the existing Search service |
| Query the knowledge base | Search Index Data Reader, or Search Index Data Contributor, on Search |
| Search calls the existing chat deployment | Search's managed identity needs Cognitive Services User, or equivalent inference access, on the parent Foundry resource |
| Web IQ searches | A valid Web IQ profile API key in `WEBIQ_API_KEY` |

`WORKSHOP_PREFIX` controls the names of this notebook's separate resources. It does not enforce access isolation. Source-authoring access must be restricted: authorized queries consume the stored key's shared Web IQ quota and billing.

## 1. Configuration

Use the existing workshop Search service, Foundry chat deployment and Web IQ profile. Complete Azure CLI sign-in in the configured tenant before running the notebook.

Load `.env` and `.env.azure` from the workshop directory. No APIM, broker app, embedding deployment, Storage or SharePoint settings are needed. The key and bearer tokens are never printed.

`AzureCliCredential` normally allows 10 seconds for the Azure CLI subprocess. This notebook allows 60 seconds for slow startup or token refresh. A `TimeoutExpired` / `Failed to invoke the Azure CLI` error occurs before the Search or Web IQ request, not because of the Web IQ key.

If it still times out, test token acquisition in a terminal, replacing the tenant placeholder with `AZURE_TENANT_ID` from the configuration:

```powershell
az account get-access-token --tenant '<tenant-id>' --resource https://search.azure.com --query expiresOn --output tsv
```

This prints only the expiry, not the token. If sign-in needs refreshing, run `az login --tenant '<tenant-id>'`. After reloading this notebook's updated code, rerun Configuration to recreate the credential and token provider before retrying the failed cell.

In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_AI_ACCOUNT_NAME',
            'AZURE_AI_MODEL_DEPLOYMENT_NAME', 'WEBIQ_API_KEY')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Resource prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
aoai_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
chat_deployment = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
chat_model = os.getenv('AZURE_AI_MODEL_NAME', 'gpt-5-mini')
webiq_api_key = os.environ['WEBIQ_API_KEY']
api_version = '2026-08-01-preview'
webiq_source_name = f'{prefix}-nb14-webiq'
knowledge_base_name = f'{prefix}-nb14-kb'

credential = AzureCliCredential(tenant_id=tenant_id, process_timeout=60)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
print('Search:', search_endpoint)
print('Web IQ source:', webiq_source_name)
print('Knowledge base:', knowledge_base_name)

### Check HTTPS connectivity from this kernel

For a repeated `ConnectionResetError` / `WinError 10054`, run the next cell in the same kernel that failed. It makes a read-only request without supplying a Search bearer token or Web IQ key. It does not create resources or invoke Web IQ.

An HTTP response, including the expected unauthenticated 401/403, means HTTPS completed for this check; it does not prove a later retrieve request will succeed. A connection error means this kernel could not obtain a usable HTTP response. A successful terminal or separate-process test does not rule out a repeated failure here.

Compare the interpreter and proxy/CA flags with the intended environment. A reset can originate from the endpoint or an intervening network/security component; this check alone cannot identify which one. Do not disable certificate verification or bypass organizational proxy/network controls.

In [ ]:
import sys
from requests.exceptions import RequestException

print('UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.executable)
print('Search:', search_endpoint)
proxies = requests.utils.get_environ_proxies(search_endpoint)
print('Environment proxy used:', any(name in proxies for name in ('http', 'https', 'all')))
print('Custom CA bundle configured:', bool(os.getenv('REQUESTS_CA_BUNDLE') or os.getenv('CURL_CA_BUNDLE')))
try:
    connectivity_response = requests.get(
        f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
        timeout=(10, 20), allow_redirects=False,
    )
    print('HTTPS completed; HTTP status:', connectivity_response.status_code)
    print('This unauthenticated check does not validate Search permissions or Web IQ.')
except RequestException as exc:
    print('HTTPS check failed:', type(exc).__name__)
    print('No usable HTTP response. The network/TLS failure is not resolved in this kernel.')

## 2. Create the Web IQ source

Running this cell stores `WEBIQ_API_KEY` in the new source's `storedHeaders` authentication. It does not change sources used by notebooks 09 or 10. Rerunning updates this source, including its stored key.

Only the basic `web` tool is configured. `outputParsing: auto` is Search's response parser, not the Web IQ Auto tool. Passage-format guidance avoids full-document formats that the workshop profile previously rejected. This is planner guidance, not a server-side parameter override.

In [ ]:
webiq_source = {
    'name': webiq_source_name,
    'kind': 'mcpServer',
    'description': (
        'Basic Web IQ web search for public information. For every web call, explicitly set '
        'contentFormat to passage; use query-dependent passages, not text, html or markdown.'
    ),
    'resultsProcessing': 'rerank',
    'mcpServerParameters': {
        'serverURL': 'https://api.microsoft.ai/v3/mcp',
        'authentication': {
            'kind': 'storedHeaders',
            'storedHeadersParameters': {'headers': {'x-apikey': webiq_api_key}},
        },
        'tools': [{'name': 'web', 'outputParsing': {'kind': 'auto'}, 'maxOutputTokens': 4000}],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=webiq_source, timeout=(10, 120),
)
response.raise_for_status()
print('Web IQ source ready:', webiq_source_name, '| authentication: storedHeaders | tool: web')

## 3. Create the Web IQ-only knowledge base

This knowledge base references only the source above. The existing chat model plans the web query and synthesizes the answer. No indexes, ingestion or other knowledge sources are involved.

In [ ]:
knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Standalone public web retrieval through basic Web IQ search only.',
    'retrievalInstructions': (
        f'Use {webiq_source_name} to retrieve public web evidence for the question. '
        'For every web call explicitly set contentFormat to passage. '
        'Use specific search terms, locations and dates from the question. '
        'For current software releases prefer the official release or downloads page. '
        'For weather, search for the requested location and forecast date.'
    ),
    'answerInstructions': (
        'Write a concise final answer from the retrieved passages and cite supporting reference IDs. '
        'Treat MCP web passages as evidence already retrieved for this request. '
        'If evidence does not establish the requested fact, identify the missing information. '
        'Distinguish forecast dates from page crawl dates and stable releases from prereleases. '
        'Do not infer the latest release from an incomplete list of older versions.'
    ),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': webiq_source_name}],
    'models': [{
        'kind': 'azureOpenAI',
        'azureOpenAIParameters': {
            'resourceUri': aoai_endpoint,
            'deploymentId': chat_deployment,
            'modelName': chat_model,
        },
    }],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Web IQ-only knowledge base ready:', knowledge_base_name)

## 4. Choose a question and output mode

The default is an explicitly located, dated weather question. Replace it with another public question, such as `What is the latest stable Python release and its release date, according to Python.org?`.

Start with `answerSynthesis`. Then change `output_mode` to `extractiveData` and rerun this cell and the next cell to inspect retrieval without a generated answer. The two modes make separate searches; their results can differ. This does not change the stored knowledge-base configuration.

Web search results are not a live sensor or clock. Check that retrieved passages actually support the requested place, date and fact rather than assuming a returned reference proves answer accuracy.

In [ ]:
today = datetime.now(timezone.utc).date().isoformat()
question = f'What is the weather forecast for Seattle, Washington, on {today}? Cite the forecast source and date.'
output_mode = 'answerSynthesis'
if output_mode not in ('answerSynthesis', 'extractiveData'):
    raise ValueError('Use answerSynthesis or extractiveData.')
print('Question:', question)
print('Output mode:', output_mode)

In [ ]:
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'outputMode': output_mode,
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [{
            'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
            'includeReferences': True, 'includeReferenceSourceData': True,
        }],
    }, timeout=(10, 120),
)
print('HTTP status:', response.status_code)
print('Request ID:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
if response.status_code >= 400:
    print(response.text.replace(webiq_api_key, '<redacted>')[:8000])
response.raise_for_status()
result = response.json()

print('\nActivity:')
print(json.dumps(result.get('activity', []), indent=2).replace(webiq_api_key, '<redacted>'))
print('\nAnswer or extractive output:')
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text', ''))
print('\nReferences and retrieved passages:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    try:
        parsed = json.loads(source_data.get('content') or '{}')
        if isinstance(parsed, dict):
            source_data = {**source_data, **parsed}
    except (TypeError, ValueError):
        pass
    print('-', reference.get('id'), source_data.get('title', ''), source_data.get('url', ''))
    print(str(source_data.get('content') or '')[:2000])
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or Web IQ failed. Inspect the activity above.')
if not result.get('references'):
    print('No references returned. Inspect activity; this alone does not establish an authentication failure.')

## 5. Interpret the result and try the portal

| Evidence | Interpretation |
|---|---|
| HTTP 401/403 from Search | Inspect Search sign-in, tenant, RBAC and network access; this is not automatically a Web IQ error. |
| MCP activity reports `auth_missing_token` | Web IQ did not receive accepted authentication. This source uses a stored `x-apikey`, not APIM header injection. Verify the key/profile and rerun source creation after correcting it. |
| MCP activity reports another error | Inspect the tool arguments and actual error. For a content-format rejection, check that `contentFormat` is `passage`. |
| MCP returned relevant passages, but the answer says it cannot search | The answer contradicts retrieved evidence. Inspect the passages and compare `extractiveData`; do not diagnose a missing key from answer wording alone. |
| MCP returned no relevant evidence | Check the search terms, location and date. HTTP 200 or a result count does not prove the question was answered. |

In the Foundry IQ portal, select this Search service and the printed `WORKSHOP_PREFIX`-`nb14-kb` knowledge base. When selecting a source for another knowledge base, use this notebook's `WORKSHOP_PREFIX`-`nb14-webiq` source, not the keyless `iq-workshop-webiq` APIM source.

The key belongs to the knowledge source, not to a knowledge base or chat. It is shared by authorized callers. Rotate it in Web IQ and rerun source creation to update the stored value; revoke it when no longer needed. Do not print `webiq_source`, which contains the key.

## 6. Optional cleanup

Leave cleanup disabled while testing in the portal. Enabling it deletes only this notebook's knowledge base and source, in that order. It does not remove the shared Search service, model deployment, Web IQ profile/key, or resources from notebooks 09 and 10.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in (('knowledgebases', knowledge_base_name),
                             ('knowledgesources', webiq_source_name)):
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Removed or already absent:', name)
else:
    print('Cleanup disabled; Web IQ-only resources remain available.')

## References

- [MCP server knowledge sources and authentication](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve action and response](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)
- [Answer synthesis and per-request output mode](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)